In [ ]:
! git clone https://github.com/mmeagher/point-e

In [ ]:
%cd point-e/

In [ ]:
pip install -e .

In [ ]:
import torch
from tqdm.auto import tqdm

import point_e.util.point_cloud as pt

from point_e.diffusion.configs import DIFFUSION_CONFIGS, diffusion_from_config
from point_e.diffusion.sampler import PointCloudSampler
from point_e.models.download import load_checkpoint
from point_e.models.configs import MODEL_CONFIGS, model_from_config
from point_e.util.plotting import plot_point_cloud

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print('creating base model...')
base_name = 'base40M-textvec'
base_model = model_from_config(MODEL_CONFIGS[base_name], device)
base_model.eval()
base_diffusion = diffusion_from_config(DIFFUSION_CONFIGS[base_name])

print('creating upsample model...')
upsampler_model = model_from_config(MODEL_CONFIGS['upsample'], device)
upsampler_model.eval()
upsampler_diffusion = diffusion_from_config(DIFFUSION_CONFIGS['upsample'])

print('downloading base checkpoint...')
base_model.load_state_dict(load_checkpoint(base_name, device))

print('downloading upsampler checkpoint...')
upsampler_model.load_state_dict(load_checkpoint('upsample', device))

In [ ]:
import numpy as np
import pandas as pd
from datetime import datetime
import os

# Create output directory for point clouds
output_dir = 'pointcloud_variations'
os.makedirs(output_dir, exist_ok=True)

# Define prompt variations
prompts = [
    'furniture for dreaming whose surface and form is shaped by the emotion of fear, sadness and anxiety and which involves loved ones in a setting that is very real',
    'a surface for dreaming whose form is shaped by the emotion of fear, sadness and anxiety and which involves loved ones in a setting that is very real',
    'a surface for dreaming whose form is shaped by the emotion of gladness and delight and which involves pets in a setting that is unfamiliar and unreal'
]

# Define parameter ranges for variations
guidance_scales = [2.0, 2.5, 3.0, 3.5, 4.0, 4.5, 5.0]
karras_steps_options = [64, 96, 128]
s_churn_options = [0.5, 1.0, 2.0, 3.0, 5.0]

# Initialize tracking dataframe
tracking_data = []

print(f'Setup complete. Will generate 100 variations.')

In [ ]:
# Generate 10 variations
num_variations = 10

# Store point clouds in memory for visualization
point_clouds = []

print(f'Starting generation of {num_variations} variations...')
print(f'This may take a while. Progress will be shown below.')

for i in range(num_variations):
    variation_id = i + 1

    # Select parameters for this variation
    prompt = prompts[i % len(prompts)]
    guidance_scale = guidance_scales[i % len(guidance_scales)]
    karras_steps = karras_steps_options[(i // len(prompts)) % len(karras_steps_options)]
    s_churn_base = s_churn_options[(i // (len(prompts) * len(karras_steps_options))) % len(s_churn_options)]
    s_churn_upsample = s_churn_options[(i // len(guidance_scales)) % len(s_churn_options)] * 0.2

    print(f'\n--- Variation {variation_id}/{num_variations} ---')
    print(f'Prompt: {prompt[:60]}...' if len(prompt) > 60 else f'Prompt: {prompt}')
    print(f'Parameters: guidance={guidance_scale}, steps={karras_steps}, churn={s_churn_base:.1f}')

    # Create sampler with these parameters
    sampler = PointCloudSampler(
        device=device,
        models=[base_model, upsampler_model],
        diffusions=[base_diffusion, upsampler_diffusion],
        num_points=[1024, 4096 - 1024],
        aux_channels=['R', 'G', 'B'],
        guidance_scale=[guidance_scale, 0.0],
        model_kwargs_key_filter=('texts', ''),
        karras_steps=[karras_steps, karras_steps],
        s_churn=[s_churn_base, s_churn_upsample],
    )

    # Generate point cloud
    samples = None
    for x in tqdm(sampler.sample_batch_progressive(batch_size=1, model_kwargs=dict(texts=[prompt])),
                  desc=f'Generating {variation_id}'):
        samples = x

    # Convert to point cloud
    pc = sampler.output_to_point_clouds(samples)[0]

    # Apply color based on height for consistency
    z_coords = pc.coords[:, 2]
    z_min, z_max = z_coords.min(), z_coords.max()
    if z_max > z_min:
        normalized_z = (z_coords - z_min) / (z_max - z_min)
    else:
        normalized_z = np.zeros_like(z_coords)

    pc.channels['R'] = normalized_z
    pc.channels['G'] = 0.3 * np.ones_like(normalized_z)
    pc.channels['B'] = 1.0 - normalized_z

    # Store point cloud in memory for visualization
    point_clouds.append(pc)

    # Save point cloud to PLY file
    filename = f'pointcloud_{variation_id:03d}.ply'
    filepath = os.path.join(output_dir, filename)
    with open(filepath, 'wb') as f:
        pc.write_ply(f)

    # Record parameters
    tracking_data.append({
        'id': variation_id,
        'prompt': prompt,
        'guidance_scale_base': guidance_scale,
        'guidance_scale_upsample': 0.0,
        'karras_steps_base': karras_steps,
        'karras_steps_upsample': karras_steps,
        's_churn_base': s_churn_base,
        's_churn_upsample': s_churn_upsample,
        'num_points': 4096,
        'filename': filename,
        'timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    })

    print(f'Saved: {filepath}')

print(f'\n✓ All {num_variations} variations generated successfully!')
print(f'✓ Point clouds stored in memory for visualization')

In [ ]:
# Generate meshes from point clouds
from point_e.util.pc_to_mesh import marching_cubes_mesh

# Create output directory for meshes
mesh_output_dir = 'mesh_variations'
os.makedirs(mesh_output_dir, exist_ok=True)

print(f'\nLoading SDF model for mesh generation...')
sdf_name = 'sdf'
sdf_model = model_from_config(MODEL_CONFIGS[sdf_name], device)
sdf_model.eval()
sdf_model.load_state_dict(load_checkpoint(sdf_name, device))

print(f'\nGenerating meshes from {len(point_clouds)} point clouds...')
print('This may take a while. Progress will be shown below.')

meshes = []

for i, pc in enumerate(point_clouds):
    variation_id = i + 1
    print(f'\n--- Generating Mesh {variation_id}/{len(point_clouds)} ---')
    
    # Generate mesh using marching cubes
    mesh = marching_cubes_mesh(
        pc=pc,
        model=sdf_model,
        batch_size=4096,
        grid_size=32,  # Use 32 for faster generation, increase to 128 for higher quality
        progress=True,
    )
    
    # Store mesh in memory
    meshes.append(mesh)
    
    # Save mesh to PLY file
    mesh_filename = f'mesh_{variation_id:03d}.ply'
    mesh_filepath = os.path.join(mesh_output_dir, mesh_filename)
    with open(mesh_filepath, 'wb') as f:
        mesh.write_ply(f)
    
    print(f'Saved: {mesh_filepath}')

print(f'\n✓ All {len(meshes)} meshes generated successfully!')
print(f'✓ Meshes stored in memory and saved to {mesh_output_dir}/')

In [ ]:
# Save tracking data to CSV
df = pd.DataFrame(tracking_data)
csv_filename = 'pointcloud_parameters.csv'
df.to_csv(csv_filename, index=False)

print(f'Parameter tracking saved to: {csv_filename}')
print(f'\nFirst 10 variations:')
print(df.head(10).to_string())
print(f'\n...')
print(f'\nLast 10 variations:')
print(df.tail(10).to_string())
print(f'\nTotal variations: {len(df)}')

# Create zip file containing all PLY files (point clouds and meshes) and CSV
import zipfile

zip_filename = f'pointcloud_variations_{datetime.now().strftime("%Y%m%d_%H%M%S")}.zip'

print(f'\nCreating zip file: {zip_filename}')

with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:
    # Add CSV file
    zipf.write(csv_filename, csv_filename)
    print(f'  Added: {csv_filename}')
    
    # Add all point cloud PLY files
    ply_count = 0
    for filename in os.listdir(output_dir):
        if filename.endswith('.ply'):
            filepath = os.path.join(output_dir, filename)
            # Store with folder structure
            zipf.write(filepath, os.path.join(output_dir, filename))
            ply_count += 1
    
    print(f'  Added: {ply_count} point cloud PLY files from {output_dir}/')
    
    # Add all mesh PLY files
    mesh_count = 0
    for filename in os.listdir(mesh_output_dir):
        if filename.endswith('.ply'):
            filepath = os.path.join(mesh_output_dir, filename)
            # Store with folder structure
            zipf.write(filepath, os.path.join(mesh_output_dir, filename))
            mesh_count += 1
    
    print(f'  Added: {mesh_count} mesh PLY files from {mesh_output_dir}/')

print(f'\n✓ Zip file created successfully: {zip_filename}')
print(f'  Total size: {os.path.getsize(zip_filename) / (1024*1024):.2f} MB')
print(f'  Contents: {csv_filename}, {ply_count} point clouds, {mesh_count} meshes')

In [ ]:
# Optional: Visualize a specific variation
# Change the variation_id below to view different results
variation_to_view = 1  # Change this to view different variations (1-100)

if variation_to_view <= len(tracking_data):
    # Load the saved point cloud
    variation_file = os.path.join(output_dir, f'pointcloud_{variation_to_view:03d}.ply')

    if os.path.exists(variation_file):
        # Get parameters for this variation
        params = df[df['id'] == variation_to_view].iloc[0]

        print(f'Viewing Variation {variation_to_view}:')
        print(f'Prompt: {params["prompt"]}')
        print(f'Guidance Scale: {params["guidance_scale_base"]}')
        print(f'Karras Steps: {params["karras_steps_base"]}')
        print(f'S-Churn: {params["s_churn_base"]}')
        print(f'File: {params["filename"]}')

        # Note: To view the point cloud, you would need to load it from the PLY file
        # For now, this cell just displays the parameters
        print(f'\nPoint cloud saved at: {variation_file}')
else:
    print(f'Variation {variation_to_view} not found. Valid range: 1-{len(tracking_data)}')

In [ ]:
# Visualize all generated point clouds and meshes in a grid
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from mpl_toolkits.mplot3d.art3d import Poly3DCollection

# Each variation takes 2 columns (point cloud + mesh), so 3 variations per row
variations_per_row = 3
num_cols = variations_per_row * 2  # 6 columns total
num_rows = (len(point_clouds) + variations_per_row - 1) // variations_per_row

# Create figure with subplots
fig = plt.figure(figsize=(20, 3.5 * num_rows))

print(f'Generating visualization grid for {len(point_clouds)} variations (point clouds + meshes)...')

for idx, (pc, mesh, row) in enumerate(zip(point_clouds, meshes, tracking_data)):
    variation_id = row['id']
    
    # Calculate subplot positions (2 per variation: point cloud and mesh)
    subplot_row = idx // variations_per_row
    subplot_col_base = (idx % variations_per_row) * 2
    
    # === Point Cloud Subplot ===
    ax_pc = fig.add_subplot(num_rows, num_cols, subplot_row * num_cols + subplot_col_base + 1, projection='3d')
    
    # Plot point cloud
    coords = pc.coords
    colors = np.stack([pc.channels['R'], pc.channels['G'], pc.channels['B']], axis=1)
    
    ax_pc.scatter(coords[:, 0], coords[:, 1], coords[:, 2], 
                  c=colors, s=1, alpha=0.8)
    
    # Set title for point cloud
    prompt_short = row['prompt'][:30] + '...' if len(row['prompt']) > 30 else row['prompt']
    title_pc = f"#{variation_id} Point Cloud\n{prompt_short}\nG={row['guidance_scale_base']}, S={row['karras_steps_base']}"
    ax_pc.set_title(title_pc, fontsize=6, pad=5)
    
    # Set equal aspect ratio for point cloud
    max_range = np.array([coords[:, 0].max()-coords[:, 0].min(), 
                          coords[:, 1].max()-coords[:, 1].min(), 
                          coords[:, 2].max()-coords[:, 2].min()]).max() / 2.0
    
    mid_x = (coords[:, 0].max()+coords[:, 0].min()) * 0.5
    mid_y = (coords[:, 1].max()+coords[:, 1].min()) * 0.5
    mid_z = (coords[:, 2].max()+coords[:, 2].min()) * 0.5
    
    ax_pc.set_xlim(mid_x - max_range, mid_x + max_range)
    ax_pc.set_ylim(mid_y - max_range, mid_y + max_range)
    ax_pc.set_zlim(mid_z - max_range, mid_z + max_range)
    ax_pc.set_xticks([])
    ax_pc.set_yticks([])
    ax_pc.set_zticks([])
    ax_pc.view_init(elev=20, azim=45)
    
    # === Mesh Subplot ===
    ax_mesh = fig.add_subplot(num_rows, num_cols, subplot_row * num_cols + subplot_col_base + 2, projection='3d')
    
    # Plot mesh
    verts = mesh.verts
    faces = mesh.faces
    
    # Create mesh colors if available
    if hasattr(mesh, 'vertex_channels') and 'R' in mesh.vertex_channels:
        mesh_colors = np.stack([
            mesh.vertex_channels['R'],
            mesh.vertex_channels['G'],
            mesh.vertex_channels['B']
        ], axis=1)
    else:
        # Use default color if no vertex colors
        mesh_colors = np.ones((len(verts), 3)) * 0.7
    
    # Create 3D polygon collection
    triangles = verts[faces]
    face_colors = mesh_colors[faces].mean(axis=1)  # Average vertex colors for each face
    
    mesh_collection = Poly3DCollection(triangles, alpha=0.7, edgecolor='none')
    mesh_collection.set_facecolor(face_colors)
    ax_mesh.add_collection3d(mesh_collection)
    
    # Set title for mesh
    title_mesh = f"#{variation_id} Mesh\n{prompt_short}\nG={row['guidance_scale_base']}, S={row['karras_steps_base']}"
    ax_mesh.set_title(title_mesh, fontsize=6, pad=5)
    
    # Set equal aspect ratio for mesh (use same bounds as point cloud)
    ax_mesh.set_xlim(mid_x - max_range, mid_x + max_range)
    ax_mesh.set_ylim(mid_y - max_range, mid_y + max_range)
    ax_mesh.set_zlim(mid_z - max_range, mid_z + max_range)
    ax_mesh.set_xticks([])
    ax_mesh.set_yticks([])
    ax_mesh.set_zticks([])
    ax_mesh.view_init(elev=20, azim=45)

plt.tight_layout()
plt.savefig('pointcloud_grid_visualization.png', dpi=150, bbox_inches='tight')
print(f'\n✓ Visualization saved as: pointcloud_grid_visualization.png')
plt.show()

In [ ]:
# Alternative: View point clouds individually with full details
# Set which variations to display (or use 'all' to show everything)
variations_to_display = 'all'  # Options: 'all', or a list like [1, 3, 5, 7]

if variations_to_display == 'all':
    display_list = range(len(point_clouds))
else:
    # Convert 1-indexed to 0-indexed
    display_list = [v - 1 for v in variations_to_display]

print(f'Displaying {len(list(display_list))} point cloud(s) individually...\n')

for idx in display_list:
    if idx < len(point_clouds):
        pc = point_clouds[idx]
        row = tracking_data[idx]
        variation_id = row['id']
        
        # Display parameters
        print(f"{'='*80}")
        print(f"VARIATION #{variation_id}")
        print(f"{'='*80}")
        print(f"Prompt: {row['prompt']}")
        print(f"Guidance Scale: {row['guidance_scale_base']}")
        print(f"Karras Steps: {row['karras_steps_base']}")
        print(f"S-Churn (base): {row['s_churn_base']}")
        print(f"S-Churn (upsample): {row['s_churn_upsample']}")
        print(f"Number of points: {row['num_points']}")
        print(f"Generated: {row['timestamp']}")
        print(f"File: {row['filename']}")
        print()
        
        # Create figure with larger size for better viewing
        fig = plt.figure(figsize=(12, 8))
        ax = fig.add_subplot(111, projection='3d')
        
        # Plot point cloud
        coords = pc.coords
        colors = np.stack([pc.channels['R'], pc.channels['G'], pc.channels['B']], axis=1)
        
        ax.scatter(coords[:, 0], coords[:, 1], coords[:, 2], 
                   c=colors, s=2, alpha=0.6)
        
        # Set title
        prompt_display = row['prompt'][:80] + '...' if len(row['prompt']) > 80 else row['prompt']
        ax.set_title(f"Variation #{variation_id}: {prompt_display}", fontsize=10, pad=15)
        
        # Set equal aspect ratio
        max_range = np.array([coords[:, 0].max()-coords[:, 0].min(), 
                              coords[:, 1].max()-coords[:, 1].min(), 
                              coords[:, 2].max()-coords[:, 2].min()]).max() / 2.0
        
        mid_x = (coords[:, 0].max()+coords[:, 0].min()) * 0.5
        mid_y = (coords[:, 1].max()+coords[:, 1].min()) * 0.5
        mid_z = (coords[:, 2].max()+coords[:, 2].min()) * 0.5
        
        ax.set_xlim(mid_x - max_range, mid_x + max_range)
        ax.set_ylim(mid_y - max_range, mid_y + max_range)
        ax.set_zlim(mid_z - max_range, mid_z + max_range)
        
        ax.set_xlabel('X')
        ax.set_ylabel('Y')
        ax.set_zlabel('Z')
        
        # Set viewing angle
        ax.view_init(elev=20, azim=45)
        
        plt.tight_layout()
        plt.show()
        print()

print(f"✓ Displayed {len(list(display_list))} visualization(s)")